# USRP X310 LTE IQ Capture Notebook

This notebook opens USRP X310, tunes to a **763 MHz** LTE downlink channel, captures complex IQ samples for a **10 MHz LTE channel**, saves them to disk.

Default LTE capture parameters:
- Center frequency: `763e6` Hz
- LTE channel bandwidth: `10e6` Hz
- Sample rate: `15.36e6` samples/s, matching 10 MHz LTE with 1024-point OFDM FFT and 15 kHz subcarrier spacing
- Output: `.npz` and optional raw `.cf32`

In [ ]:
# Optional: install/check Python dependencies outside the notebook if needed:
# sudo apt install python3-uhd python3-numpy python3-scipy python3-matplotlib
# or if UHD was built from source, make sure PYTHONPATH can find the UHD Python module.

import sys
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    import uhd
    print("UHD Python module loaded:", uhd)
except Exception as e:
    print("Could not import uhd.")
    print("Fix: install UHD Python bindings or source your UHD environment.")
    raise e


## 1. Capture configuration

The X310 IP is `192.168.10.2`.

`subdev="A:0"` means motherboard slot A, daughterboard channel 0. If your daughterboard is in slot B, change it to `"B:0"`.

For many UBX/WBX/SBX-style daughterboards:
- `"RX2"` is receive-only input.
- `"TX/RX"` is shared transmit/receive input.

In [ ]:
# ---- USRP/X310 settings ----
usrp_args = "addr=192.168.10.2"  
subdev    = "A:0"                
channel   = 0
antenna   = "RX2"              

# ---- LTE channel settings ----
center_freq_hz = 763e6            
lte_channel_bw_hz = 10e6     
sample_rate_sps = 15.36e6      
rx_bandwidth_hz = 10e6           

# ---- Capture settings ----
rx_gain_db = 30               
duration_s = 0.20               
settle_time_s = 0.2           

# ---- Output settings ----
out_dir = Path("./captures")
out_dir.mkdir(exist_ok=True)

timestamp = time.strftime("%Y%m%d_%H%M%S")
base_name = f"x310_lte_763MHz_10MHz_{timestamp}"
npz_path = out_dir / f"{base_name}.npz"
cf32_path = out_dir / f"{base_name}.cf32"

print("Samples to capture:", int(duration_s * sample_rate_sps))
print("Approx memory:", int(duration_s * sample_rate_sps) * np.dtype(np.complex64).itemsize / 1e6, "MB")
print("Output NPZ:", npz_path)
print("Output CF32:", cf32_path)


## 2. Open and configure the X310

In [ ]:
usrp = uhd.usrp.MultiUSRP(usrp_args)

# Configure RX frontend
usrp.set_rx_subdev_spec(uhd.usrp.SubdevSpec(subdev))
usrp.set_rx_rate(sample_rate_sps, channel)
usrp.set_rx_freq(uhd.types.TuneRequest(center_freq_hz), channel)
usrp.set_rx_gain(rx_gain_db, channel)
usrp.set_rx_bandwidth(rx_bandwidth_hz, channel)

# Antenna selection can fail if the daughterboard does not expose that port name.
try:
    usrp.set_rx_antenna(antenna, channel)
except Exception as e:
    print(f"Warning: could not set antenna={antenna!r}. Available antennas:")
    try:
        print(usrp.get_rx_antennas(channel))
    except Exception:
        pass
    raise e

time.sleep(settle_time_s)

print("USRP configured")
print("RX rate      :", usrp.get_rx_rate(channel))
print("RX freq      :", usp_freq := usrp.get_rx_freq(channel))
print("RX gain      :", usrp.get_rx_gain(channel))
print("RX bandwidth :", usrp.get_rx_bandwidth(channel))
print("RX antenna   :", usrp.get_rx_antenna(channel))


## 3. Capture IQ samples

The simple path uses UHD's Python convenience method `recv_num_samps()`. This is the easiest way for an offline LTE capture.

For longer captures, prefer the manual streamer cell further below because it gives better visibility into metadata errors and overflows.


In [ ]:
num_samps = int(duration_s * sample_rate_sps)

# Convenience capture.
# Shape is commonly (num_channels, num_samps). We use channel list [0].
iq = usrp.recv_num_samps(
    num_samps,
    center_freq_hz,
    sample_rate_sps,
    [channel],
    rx_gain_db
)

iq = np.asarray(iq)
print("Raw returned shape:", iq.shape, iq.dtype)

# Convert to 1-D complex64 vector
if iq.ndim == 2:
    iq = iq[0]
iq = iq.astype(np.complex64, copy=False)

print("Final IQ shape:", iq.shape, iq.dtype)

## 4. Save capture

`.npz` keeps metadata together with the IQ vector. `.cf32` is useful for tools such as GNU Radio, MATLAB, or custom C/Python readers.


In [ ]:
metadata = {
    "center_freq_hz": center_freq_hz,
    "sample_rate_sps": sample_rate_sps,
    "lte_channel_bw_hz": lte_channel_bw_hz,
    "rx_bandwidth_hz": rx_bandwidth_hz,
    "rx_gain_db": rx_gain_db,
    "duration_s": duration_s,
    "usrp_args": usrp_args,
    "subdev": subdev,
    "antenna": antenna,
    "channel": channel,
}

np.savez(npz_path, iq=iq, **metadata)
iq.tofile(cf32_path)

print("Saved:")
print(" ", npz_path)
print(" ", cf32_path)


## 5. Signal checks

In [ ]:
# Time-domain magnitude
n_plot = min(len(iq), 20000)
t_ms = np.arange(n_plot) / sample_rate_sps * 1e3

plt.figure(figsize=(12, 4))
plt.plot(t_ms, np.real(iq[:n_plot]), label="I", linewidth=0.8)
plt.plot(t_ms, np.imag(iq[:n_plot]), label="Q", linewidth=0.8)
plt.xlabel("Time (ms)")
plt.ylabel("Amplitude")
plt.title("IQ time-domain preview")
plt.grid(True)
plt.legend()
plt.show()

print("Mean power:", np.mean(np.abs(iq)**2))
print("Peak magnitude:", np.max(np.abs(iq)))
print("RMS magnitude:", np.sqrt(np.mean(np.abs(iq)**2)))
print("Clipping hint: if many samples are near full-scale magnitude ~1.0, reduce RX gain.")

In [ ]:
# PSD using matplotlib Welch-style PSD
plt.figure(figsize=(12, 4))
plt.psd(iq, NFFT=4096, Fs=sample_rate_sps, Fc=center_freq_hz, scale_by_freq=True)
plt.title("Power spectral density")
plt.xlabel("Frequency (Hz)")
plt.ylabel("PSD")
plt.grid(True)
plt.show()

In [ ]:
# Spectrogram
plt.figure(figsize=(12, 5))
plt.specgram(iq[:min(len(iq), int(0.1 * sample_rate_sps))], NFFT=1024, Fs=sample_rate_sps, Fc=center_freq_hz, noverlap=768)
plt.title("Spectrogram preview")
plt.xlabel("Time (s)")
plt.ylabel("Frequency (Hz)")
plt.colorbar(label="Power")
plt.show()


## 6. Load a saved capture later

In [ ]:
# Example load:
# data = np.load(npz_path)
# iq_loaded = data["iq"]
# fs_loaded = float(data["sample_rate_sps"])
# fc_loaded = float(data["center_freq_hz"])
# print(iq_loaded.shape, fs_loaded, fc_loaded)
